<a href="https://colab.research.google.com/github/iramme/SONAR_TOKEN/blob/main/final_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Final Model Evaluation

## 1. Importing Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Imbalanced-learn
from imblearn.over_sampling import SMOTE


In [ ]:
df = pd.read_csv(
    r"C:\Users\Bdrsk\Downloads\balanced_data1.csv"
)

In [ ]:
print("Distribution initiale:")
print(df['sentiment'].value_counts())

Distribution initiale:
sentiment
positive    139110
neutral      84734
negative     72177
Name: count, dtype: int64


In [ ]:
# Supprimer les doublons sur la colonne 'clean_text', garder la première occurrence
df = df.drop_duplicates(subset=['clean_text'], keep='first').reset_index(drop=True)

# Vérification
print("Après suppression des doublons :")
print(df['sentiment'].value_counts())
print("Nombre de doublons restant :", df['clean_text'].duplicated().sum())


Après suppression des doublons :
sentiment
positive    139110
neutral      78244
negative     72177
Name: count, dtype: int64
Nombre de doublons restant : 0


In [ ]:
df.to_csv("balanced_data2.csv")

## 2. Splitting the Dataset

In [ ]:

X_text = df["clean_text"]
y = df["sentiment"]

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)




## 3. Text Vectorization using TF-IDF

In [ ]:

vectorizer = TfidfVectorizer(
    ngram_range=(1,2),
    max_features=30000,
    min_df=5
)

X_train_vec = vectorizer.fit_transform(X_train_text)
X_test_vec = vectorizer.transform(X_test_text)





## 4. Handling Class Imbalance with SMOTE

Utilisation de SMOTE pour équilibrer toutes les classes (negative, neutral, positive) à la taille de la classe majoritaire.

Permet d’améliorer la performance sur les classes minoritaires, notamment la classe neutral.

SMOTE crée de nouveaux exemples artificiels pour les classes sous-représentées.

Ici, on équilibre toutes les classes pour qu’elles aient le même nombre d’exemples que la classe majoritaire.

fit_resample() génère le nouveau jeu de données équilibré (X_train_smote, y_train_smote).

Pourquoi c’est utile :

Le modèle apprend également bien les classes minoritaires.

Les performances sur la classe neutral s’améliorent.

On réduit le biais du modèle vers la classe majoritaire.

Important : SMOTE ne doit être appliqué que sur les données d’entraînement, jamais sur le test.

In [ ]:

# taille cible = classe majoritaire
target_size = y_train.value_counts().max()

smote = SMOTE(
    sampling_strategy={
        "negative": target_size,
        "neutral": target_size,
        "positive": target_size
    },
    random_state=42
)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train_vec, y_train
)


## 5. Train Model LogisticRegression

In [ ]:

model = LogisticRegression(
    max_iter=1000,
    n_jobs=-1
)

model.fit(X_train_smote, y_train_smote)


LogisticRegression(max_iter=1000, n_jobs=-1)

In [ ]:

y_pred = model.predict(X_test_vec)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))


              precision    recall  f1-score   support

    negative       0.83      0.85      0.84     14436
     neutral       0.75      0.78      0.77     15649
    positive       0.93      0.90      0.92     27822

    accuracy                           0.86     57907
   macro avg       0.84      0.84      0.84     57907
weighted avg       0.86      0.86      0.86     57907

[[12263  1900   273]
 [ 1809 12259  1581]
 [  693  2100 25029]]


## 5. Train Model LinearSVC

In [ ]:

model = LinearSVC()

model.fit(X_train_smote, y_train_smote)

y_pred = model.predict(X_test_vec)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))


              precision    recall  f1-score   support

    negative       0.85      0.87      0.86     14436
     neutral       0.77      0.82      0.79     15649
    positive       0.93      0.90      0.92     27822

    accuracy                           0.87     57907
   macro avg       0.85      0.86      0.86     57907
weighted avg       0.87      0.87      0.87     57907

[[12528  1628   280]
 [ 1382 12775  1492]
 [  761  2089 24972]]
